In [75]:
#Read all data [194 users' oversampled data]
import csv
import pandas as pd
dataset=pd.read_csv('dataset/AllOversampledNData.csv')
dataset.head()

,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,...,F57,F58,F59,F60,F61,F62,F63,F64,F65,ID
0,0.041,0.026,0.016,0.018,0.060,0.080,0.052,0.049,0.029,0.021,...,0.697,0.698,0.698,0.694,0.056,0.039,0.0,0.009,0.043,A3MC5OA9RXOOFH
1,0.041,0.026,0.016,0.018,0.060,0.080,0.052,0.049,0.029,0.021,...,0.697,0.698,0.698,0.694,0.056,0.039,0.0,0.009,0.043,A3MC5OA9RXOOFH
2,0.040,0.027,0.118,0.068,0.052,0.010,0.021,0.013,0.046,0.064,...,0.677,0.677,0.684,0.684,0.105,0.031,0.0,0.008,0.060,A3MC5OA9RXOOFH
3,0.040,0.027,0.118,0.068,0.052,0.010,0.021,0.013,0.046,0.064,...,0.677,0.677,0.684,0.684,0.105,0.031,0.0,0.008,0.060,A3MC5OA9RXOOFH
4,0.021,0.011,0.006,0.011,0.044,0.022,0.011,0.048,0.047,0.080,...,0.667,0.667,0.672,0.677,0.067,0.020,1.0,0.014,0.010,A3MC5OA9RXOOFH


In [76]:
#replace the user ID by class name and count the number of sample in each class
dataset['ID'] = pd.factorize(dataset['ID'])[0]
dataset.groupby(['ID'])['ID'].count()

ID
0      1000
1      1000
2      1000
3      1000
4      1000
       ... 
188    1000
189    1000
190    1000
191    1000
192    1000
Name: ID, Length: 193, dtype: int64

In [77]:
#seperate the profile in two groups: (i) Training profile (0-95), and (ii) auxiliary profile (96-193)
totalUser= len(pd.unique(dataset['ID']))
trainingData = dataset[dataset['ID'] < int(totalUser/2)]
auxilaryData = dataset[dataset['ID'] >= int(totalUser/2)]
print("Total user in training dataset:", len(pd.unique(trainingData['ID'])))
print("Total user in auxiliary dataset:", len(pd.unique(auxilaryData['ID'])))

Total user in training dataset: 96
Total user in auxiliary dataset: 97


In [84]:
#value range of training data
print(trainingData.max())

F1      0.243
F2      0.233
F3      0.229
F4      0.239
F5      0.234
        ...  
F62     0.246
F63     1.000
F64     0.033
F65     0.125
ID     95.000
Length: 66, dtype: float64


In [78]:
#Random project the auxiliary dataset
import numpy as np
from sklearn.random_projection import SparseRandomProjection

column1=['RPF1','RPF2','RPF3','RPF4','RPF5','RPF6','RPF7','RPF8','RPF9','RPF10','RPF11','RPF12','RPF13','RPF14','RPF15',
         'RPF16','RPF17','RPF18','RPF19','RPF20','RPF21','RPF22','RPF23','RPF24','RPF25','RPF26','RPF27','RPF28','RPF29','RPF30',
         'RPF31','RPF32','RPF33','RPF34','RPF35','RPF36','RPF37','RPF38','RPF39','RPF40','RPF41','RPF42','RPF43','RPF44','RPF45',
         'RPF46','RPF47','RPF48','RPF49','RPF50','RPF51','RPF52','RPF53','RPF54','RPF55','RPF56','ID']
column2=column1=['RPF1','RPF2','RPF3','RPF4','RPF5','RPF6','RPF7','RPF8','RPF9','RPF10','RPF11','RPF12','RPF13','RPF14','RPF15',
         'RPF16','RPF17','RPF18','RPF19','RPF20','RPF21','RPF22','RPF23','RPF24','RPF25','RPF26','RPF27','RPF28','RPF29','RPF30',
         'RPF31','RPF32','RPF33','RPF34','RPF35','RPF36','RPF37','RPF38','RPF39','RPF40','RPF41','RPF42','RPF43','RPF44','RPF45',
         'RPF46','RPF47','RPF48','RPF49','RPF50','RPF51','RPF52','RPF53','RPF54','RPF55','RPF56']
auxilaryDataRP = pd.DataFrame(columns=column1)
for seed in range(96,193):
    rng = np.random.RandomState(seed)
    X = auxilaryData[auxilaryData['ID']==seed]
    transformer = SparseRandomProjection(n_components=56, random_state=rng)
    Xdata=X.drop(columns=['ID'])
    XRP = pd.DataFrame(transformer.fit_transform(Xdata),columns=column2)
    XRP['ID']=seed
    auxilaryDataRP = pd.concat([auxilaryDataRP, XRP], ignore_index=True)
    #print("Shape of Actual data:",Xdata.shape)
    #print("Shape of Randome Matrix:", transformer.components_.shape[1],transformer.components_.shape[0])
    #print("Shape of Projected data:", X_new.shape)
print(auxilaryData.shape)
print(auxilaryDataRP.shape)


(97000, 66)
(97000, 57)


In [86]:
#Prepare the traning data for training and testing the attacker model
import tensorflow
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

Xdata=auxilaryData.drop(columns=['ID'])
XRPdata=auxilaryDataRP.drop(columns=['ID'])


Xtrain, Xtest, XRPtrain, XRPtest = train_test_split(Xdata, XRPdata, test_size=0.2, random_state=22)
Xtrain, Xval, XRPtrain, XRPval = train_test_split(Xtrain, XRPtrain, test_size=0.3, random_state=22)

In [87]:
print(Xtrain.shape)
print(XRPtrain.shape)
print(Xtest.shape)
print(XRPtest.shape)
print(Xval.shape)
print(XRPval.shape)

(54320, 65)
(54320, 56)
(19400, 65)
(19400, 56)
(23280, 65)
(23280, 56)


In [88]:
# import all necessary package for a neural network
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#matplotlib inlineimport keras
from keras.layers import Dense, Dropout, Input,Activation,Dropout, Flatten
from keras.models import Model,Sequential
from keras.datasets import mnist
#from tqdm import tqdm
#from keras.layers.advanced_activations import LeakyReLU
from keras.layers import BatchNormalization
from keras.optimizers import Adam
#import torch.nn.functional as F

In [89]:
#define optimizers for neural network
from keras.optimizers import SGD, RMSprop, Adam
def adam_optimizer():
    return Adam(lr=0.0002, beta_1=0.5)

def RMSprop_optimizer():
    return RMSprop(lr=0.001, rho=0.9)

In [90]:
#neural network architecture for model training

def create_Regressor(release=False,outDim=65):
  classifier = Sequential()
  classifier.add(Dense(128, input_dim=56))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))

  #classifier.add(Dense(256))
  #classifier.add(BatchNormalization())
  #classifier.add(Activation('relu'))

  classifier.add(Dense(256))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))

  classifier.add(Dense(256))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))

  #classifier.add(Dense(256))
  #classifier.add(BatchNormalization())
  #classifier.add(Activation('relu'))

  classifier.add(Dense(128))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))

  #if release:
  classifier.add(Dense(outDim, activation='sigmoid'))
  #else:
  #   classifier.add(Dense(Tuser))
  #np.log_softmax_v2(a, axis=axis)
  #classifier.add(F.softmax(a, dim=1))

  classifier.compile(loss='mean_squared_error', optimizer='SGD',metrics='mean_squared_error')
  return classifier

Clasf=create_Regressor()
Clasf.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_5 (Dense)             (None, 128)               7296      
                                                                 
 batch_normalization_4 (Batc  (None, 128)              512       
 hNormalization)                                                 
                                                                 
 activation_4 (Activation)   (None, 128)               0         
                                                                 
 dense_6 (Dense)             (None, 256)               33024     
                                                                 
 batch_normalization_5 (Batc  (None, 256)              1024      
 hNormalization)                                                 
                                                                 
 activation_5 (Activation)   (None, 256)              

In [91]:
#Train the classifier seperately for black-box attack
import keras

from keras.datasets import mnist
from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, UpSampling2D
from keras.optimizers import SGD, RMSprop, Adam
from keras.callbacks import ReduceLROnPlateau


learning_rate_reduction = ReduceLROnPlateau(monitor='val_acc', patience=5, verbose=1, factor=0.5,min_lr=0.0001)
callbacks_list = [learning_rate_reduction]

Regressor= create_Regressor(True,65)

#------Comment will start from here
lossc='mean_squared_error'
optimizerc=RMSprop(lr=0.001, rho=0.9)
Regressor.compile(loss=lossc, optimizer=optimizerc,metrics='mean_squared_error')
#------Comments will end from here
Rhistoryc2 =  Regressor.fit(XRPtrain, Xtrain, batch_size=64, epochs=50, validation_data=(XRPval, Xval),verbose=1, callbacks=callbacks_list)

Epoch 1/50


c:\Users\mdmor\OneDrive\Documents\GitHub\ModelDataProtection\.venv\Lib\site-packages\keras\optimizers\legacy\rmsprop.py:143: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


849/849 [==============================] - 5s 5ms/step - loss: 0.0083 - mean_squared_error: 0.0083 - val_loss: 0.0039 - val_mean_squared_error: 0.0039 - lr: 0.0010
Epoch 2/50
849/849 [==============================] - 4s 5ms/step - loss: 0.0039 - mean_squared_error: 0.0039 - val_loss: 0.0033 - val_mean_squared_error: 0.0033 - lr: 0.0010
Epoch 3/50
849/849 [==============================] - 3s 4ms/step - loss: 0.0034 - mean_squared_error: 0.0034 - val_loss: 0.0030 - val_mean_squared_error: 0.0030 - lr: 0.0010
Epoch 4/50
849/849 [==============================] - 7s 8ms/step - loss: 0.0031 - mean_squared_error: 0.0031 - val_loss: 0.0026 - val_mean_squared_error: 0.0026 - lr: 0.0010
Epoch 5/50
849/849 [==============================] - 7s 8ms/step - loss: 0.0028 - mean_squared_error: 0.0028 - val_loss: 0.0025 - val_mean_squared_error: 0.0025 - lr: 0.0010
Epoch 6/50
849/849 [==============================] - 8s 10ms/step - loss: 0.0027 - mean_squared_error: 0.0027 - val_loss: 0.0024 - val_

In [92]:
#Performance of the classifier
loss, accuracy = Regressor.evaluate(XRPtest, Xtest)
#print('Test score:', score)
print('Loss:', loss)
print('Accuracy:', accuracy)

607/607 [==============================] - 1s 2ms/step - loss: 0.0013 - mean_squared_error: 0.0013
Loss: 0.001285822014324367
Accuracy: 0.001285822014324367


In [93]:
#Random project data of original trained model
import numpy as np
from sklearn.random_projection import SparseRandomProjection

column1=['RPF1','RPF2','RPF3','RPF4','RPF5','RPF6','RPF7','RPF8','RPF9','RPF10','RPF11','RPF12','RPF13','RPF14','RPF15',
         'RPF16','RPF17','RPF18','RPF19','RPF20','RPF21','RPF22','RPF23','RPF24','RPF25','RPF26','RPF27','RPF28','RPF29','RPF30',
         'RPF31','RPF32','RPF33','RPF34','RPF35','RPF36','RPF37','RPF38','RPF39','RPF40','RPF41','RPF42','RPF43','RPF44','RPF45',
         'RPF46','RPF47','RPF48','RPF49','RPF50','RPF51','RPF52','RPF53','RPF54','RPF55','RPF56','ID']
column2=column1=['RPF1','RPF2','RPF3','RPF4','RPF5','RPF6','RPF7','RPF8','RPF9','RPF10','RPF11','RPF12','RPF13','RPF14','RPF15',
         'RPF16','RPF17','RPF18','RPF19','RPF20','RPF21','RPF22','RPF23','RPF24','RPF25','RPF26','RPF27','RPF28','RPF29','RPF30',
         'RPF31','RPF32','RPF33','RPF34','RPF35','RPF36','RPF37','RPF38','RPF39','RPF40','RPF41','RPF42','RPF43','RPF44','RPF45',
         'RPF46','RPF47','RPF48','RPF49','RPF50','RPF51','RPF52','RPF53','RPF54','RPF55','RPF56']
trainingDataRP = pd.DataFrame(columns=column1)
for seed in range(0,96):
    rng = np.random.RandomState(seed)
    X = trainingData[trainingData['ID']==seed]
    transformer = SparseRandomProjection(n_components=56, random_state=rng)
    Xdata=X.drop(columns=['ID'])
    XRP = pd.DataFrame(transformer.fit_transform(Xdata),columns=column2)
    XRP['ID']=seed
    trainingDataRP = pd.concat([trainingDataRP, XRP], ignore_index=True)
    #print("Shape of Actual data:",Xdata.shape)
    #print("Shape of Randome Matrix:", transformer.components_.shape[1],transformer.components_.shape[0])
    #print("Shape of Projected data:", X_new.shape)
print(trainingData.shape)
print(trainingDataRP.shape)

(97999, 66)
(97999, 57)


In [94]:
#Prediction plain data of original trained model
tDataRP=trainingDataRP.drop(columns=['ID'])
tDataReg= Regressor.predict(tDataRP)
print(tDataReg.shape)

3063/3063 [==============================] - 4s 1ms/step
(97999, 65)


In [95]:
#load the trained classifier
from keras.models import load_model
keras.backend.clear_session()
filename = 'Output/TrainedClassifier.h5'
#Classfier2.save(filename)
TrainedClassifier=load_model(filename)

In [96]:
#Add id with recovered data
print(type(tDataReg))
print(type(trainingDataRP['ID'].to_numpy()))
traningdataReg = pd.concat([pd.DataFrame(tDataReg), trainingDataRP['ID'].to_frame()], axis=1)

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [97]:
print(traningdataReg.shape)

(97999, 66)


In [98]:
#Random project the recover data to impersonate the user. Used different seed
import numpy as np
from sklearn.random_projection import SparseRandomProjection
#traningdataReg = pd.concat([tDataReg, trainingDataRP['ID']], axis=1)
column1=['RPF1','RPF2','RPF3','RPF4','RPF5','RPF6','RPF7','RPF8','RPF9','RPF10','RPF11','RPF12','RPF13','RPF14','RPF15',
         'RPF16','RPF17','RPF18','RPF19','RPF20','RPF21','RPF22','RPF23','RPF24','RPF25','RPF26','RPF27','RPF28','RPF29','RPF30',
         'RPF31','RPF32','RPF33','RPF34','RPF35','RPF36','RPF37','RPF38','RPF39','RPF40','RPF41','RPF42','RPF43','RPF44','RPF45',
         'RPF46','RPF47','RPF48','RPF49','RPF50','RPF51','RPF52','RPF53','RPF54','RPF55','RPF56','ID']
column2=column1=['RPF1','RPF2','RPF3','RPF4','RPF5','RPF6','RPF7','RPF8','RPF9','RPF10','RPF11','RPF12','RPF13','RPF14','RPF15',
         'RPF16','RPF17','RPF18','RPF19','RPF20','RPF21','RPF22','RPF23','RPF24','RPF25','RPF26','RPF27','RPF28','RPF29','RPF30',
         'RPF31','RPF32','RPF33','RPF34','RPF35','RPF36','RPF37','RPF38','RPF39','RPF40','RPF41','RPF42','RPF43','RPF44','RPF45',
         'RPF46','RPF47','RPF48','RPF49','RPF50','RPF51','RPF52','RPF53','RPF54','RPF55','RPF56']
traningdataReg.columns=dataset.columns
trainingDataRPReg = pd.DataFrame(columns=column1)
for seed in range(0,96):
    rng = np.random.RandomState(seed+10)
    X = traningdataReg[traningdataReg['ID']==seed]
    transformer = SparseRandomProjection(n_components=56, random_state=rng)
    Xdata=X.drop(columns=['ID'])
    XRP = pd.DataFrame(transformer.fit_transform(Xdata),columns=column2)
    XRP['ID']=seed
    trainingDataRPReg = pd.concat([trainingDataRPReg, XRP], ignore_index=True)
    #print("Shape of Actual data:",Xdata.shape)
    #print("Shape of Randome Matrix:", transformer.components_.shape[1],transformer.components_.shape[0])
    #print("Shape of Projected data:", X_new.shape)
print(traningdataReg.shape)
print(trainingDataRPReg.shape)

(97999, 66)
(97999, 57)


In [99]:
print("Total user in test dataset:", len(pd.unique(trainingDataRPReg['ID'])))

Total user in test dataset: 96


In [100]:
#Performance of the attacker by using the random projected recover data
#UserModel.compile(loss='categorical_crossentropy', optimizer=Adam(),metrics=['accuracy'])
Xtest=trainingDataRPReg.drop(columns=['ID'])
ytest=trainingDataRPReg['ID']
ytest=to_categorical(ytest)
loss, accuracy = TrainedClassifier.evaluate(Xtest,ytest)
#print('Test score:', score)
print('Loss:', loss)
print('Accuracy:', accuracy)

3063/3063 [==============================] - 4s 1ms/step - loss: 19.9066 - accuracy: 0.0000e+00
Loss: 19.906648635864258
Accuracy: 0.0


In [101]:
#Random project the recover data to impersonate the user. If the attacker know the key
import numpy as np
from sklearn.random_projection import SparseRandomProjection
#traningdataReg = pd.concat([tDataReg, trainingDataRP['ID']], axis=1)
column1=['RPF1','RPF2','RPF3','RPF4','RPF5','RPF6','RPF7','RPF8','RPF9','RPF10','RPF11','RPF12','RPF13','RPF14','RPF15',
         'RPF16','RPF17','RPF18','RPF19','RPF20','RPF21','RPF22','RPF23','RPF24','RPF25','RPF26','RPF27','RPF28','RPF29','RPF30',
         'RPF31','RPF32','RPF33','RPF34','RPF35','RPF36','RPF37','RPF38','RPF39','RPF40','RPF41','RPF42','RPF43','RPF44','RPF45',
         'RPF46','RPF47','RPF48','RPF49','RPF50','RPF51','RPF52','RPF53','RPF54','RPF55','RPF56','ID']
column2=column1=['RPF1','RPF2','RPF3','RPF4','RPF5','RPF6','RPF7','RPF8','RPF9','RPF10','RPF11','RPF12','RPF13','RPF14','RPF15',
         'RPF16','RPF17','RPF18','RPF19','RPF20','RPF21','RPF22','RPF23','RPF24','RPF25','RPF26','RPF27','RPF28','RPF29','RPF30',
         'RPF31','RPF32','RPF33','RPF34','RPF35','RPF36','RPF37','RPF38','RPF39','RPF40','RPF41','RPF42','RPF43','RPF44','RPF45',
         'RPF46','RPF47','RPF48','RPF49','RPF50','RPF51','RPF52','RPF53','RPF54','RPF55','RPF56']
traningdataReg.columns=dataset.columns
trainingDataRPReg = pd.DataFrame(columns=column1)
for seed in range(0,96):
    rng = np.random.RandomState(seed)
    X = traningdataReg[traningdataReg['ID']==seed]
    transformer = SparseRandomProjection(n_components=56, random_state=rng)
    Xdata=X.drop(columns=['ID'])
    XRP = pd.DataFrame(transformer.fit_transform(Xdata),columns=column2)
    XRP['ID']=seed
    trainingDataRPReg = pd.concat([trainingDataRPReg, XRP], ignore_index=True)
    #print("Shape of Actual data:",Xdata.shape)
    #print("Shape of Randome Matrix:", transformer.components_.shape[1],transformer.components_.shape[0])
    #print("Shape of Projected data:", X_new.shape)
print(traningdataReg.shape)
print(trainingDataRPReg.shape)

(97999, 66)
(97999, 57)


In [102]:
#Performance of the attacker by using the random projected recover data when key is known
#UserModel.compile(loss='categorical_crossentropy', optimizer=Adam(),metrics=['accuracy'])
Xtest=trainingDataRPReg.drop(columns=['ID'])
ytest=trainingDataRPReg['ID']
ytest=to_categorical(ytest)
loss, accuracy = TrainedClassifier.evaluate(Xtest,ytest)
#print('Test score:', score)
print('Loss:', loss)
print('Accuracy:', accuracy)

3063/3063 [==============================] - 4s 1ms/step - loss: 0.1201 - accuracy: 0.9698
Loss: 0.12010253220796585
Accuracy: 0.969836413860321
